# Skin Cancer Detection using Convolutional Neural Networks

This notebook trains a CNN (MobileNetV2 transfer learning) on the **HAM10000** dataset to classify skin lesions into 7 categories:

| Label | Disease |
|-------|---------|
| akiec | Actinic Keratoses |
| bcc   | Basal Cell Carcinoma |
| bkl   | Benign Keratosis |
| df    | Dermatofibroma |
| mel   | Melanoma |
| nv    | Melanocytic Nevi |
| vasc  | Vascular Lesions |

## 1. Imports & Configuration

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Add project root to path so we can import utils
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.preprocessing import (
    load_metadata, load_images_and_labels, split_data,
    get_augmented_generator, make_gradcam_heatmap,
    LABEL_MAP, LABEL_NAMES, DISEASE_NAMES, IMAGE_SIZE, NUM_CLASSES, BATCH_SIZE,
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Load Dataset

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, "dataset")
IMAGE_DIRS = [
    os.path.join(DATA_DIR, "HAM10000_images_part_1"),
    os.path.join(DATA_DIR, "HAM10000_images_part_2"),
]
METADATA_PATH = os.path.join(DATA_DIR, "HAM10000_metadata.csv")

df = load_metadata(METADATA_PATH)
print(f"Total samples in metadata: {len(df)}")
df.head()

### 2.1 Explore Class Distribution

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df, x="dx", order=df["dx"].value_counts().index, palette="viridis")
ax.set_title("Class Distribution in HAM10000")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

print("\nClass counts:")
print(df["dx"].value_counts())

### 2.2 Sample Images per Class

In [ ]:
from PIL import Image as PILImage

fig, axes = plt.subplots(1, 7, figsize=(21, 3))
for idx, cls in enumerate(LABEL_MAP.keys()):
    sample = df[df["dx"] == cls].iloc[0]
    img_id = sample["image_id"]
    for d in IMAGE_DIRS:
        p = os.path.join(d, f"{img_id}.jpg")
        if os.path.isfile(p):
            img = PILImage.open(p)
            axes[idx].imshow(img)
            axes[idx].set_title(f"{cls}\n({DISEASE_NAMES[cls][:15]}..)", fontsize=9)
            axes[idx].axis("off")
            break
plt.suptitle("Sample Image per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Preprocess Images

In [ ]:
print("Loading and resizing images to 224x224 — this may take a few minutes...")
X, y = load_images_and_labels(df, IMAGE_DIRS)
print(f"Images shape: {X.shape}  |  Labels shape: {y.shape}")

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

## 4. Data Augmentation

In [ ]:
train_gen = get_augmented_generator(X_train, y_train, batch_size=BATCH_SIZE)

# Preview a batch of augmented images
sample_imgs, sample_lbls = next(train_gen)
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for i in range(8):
    axes[i].imshow(sample_imgs[i])
    axes[i].set_title(LABEL_NAMES[int(sample_lbls[i])], fontsize=9)
    axes[i].axis("off")
plt.suptitle("Augmented Training Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Build CNN Model (Transfer Learning — MobileNetV2)

In [ ]:
# Compute class weights to handle imbalanced data
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_arr))
print("Class weights:", {LABEL_NAMES[k]: round(v, 2) for k, v in class_weights.items()})

In [ ]:
def build_model(image_size=IMAGE_SIZE, num_classes=NUM_CLASSES):
    """MobileNetV2 transfer learning model with custom classification head."""
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(*image_size, 3),
        include_top=False,
        weights="imagenet",
    )
    # Freeze the base model initially
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model()
model.summary()

## 6. Train the Model

In [ ]:
EPOCHS_PHASE1 = 15

cb_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor="val_loss", verbose=1),
]

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS_PHASE1,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=cb_list,
)

### 6.1 Fine‑tune Base Layers

In [ ]:
# Unfreeze the last 30 layers for fine-tuning
base = model.layers[0]
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_PHASE2 = 15

history_fine = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS_PHASE2,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=cb_list,
)

### 6.2 Training Curves

In [ ]:
def plot_history(h1, h2=None):
    acc = h1.history["accuracy"] + (h2.history["accuracy"] if h2 else [])
    val_acc = h1.history["val_accuracy"] + (h2.history["val_accuracy"] if h2 else [])
    loss = h1.history["loss"] + (h2.history["loss"] if h2 else [])
    val_loss = h1.history["val_loss"] + (h2.history["val_loss"] if h2 else [])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(acc, label="Train Accuracy")
    ax1.plot(val_acc, label="Val Accuracy")
    ax1.set_title("Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.legend()

    ax2.plot(loss, label="Train Loss")
    ax2.plot(val_loss, label="Val Loss")
    ax2.set_title("Loss")
    ax2.set_xlabel("Epoch")
    ax2.legend()
    plt.tight_layout()
    plt.show()

plot_history(history, history_fine)

## 7. Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
target_names = [LABEL_NAMES[i] for i in range(NUM_CLASSES)]

print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
MODEL_SAVE_PATH = os.path.join(PROJECT_ROOT, "models", "cnn_model.h5")
model.save(MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

## 9. Grad‑CAM — Explainable AI

Visualise which regions of a skin lesion the CNN focuses on.

In [ ]:
import cv2

def display_gradcam(img, heatmap, alpha=0.4):
    """Overlay heatmap on image."""
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    superimposed = heatmap_color * alpha + img
    superimposed = np.clip(superimposed, 0, 1)
    return superimposed

# Pick a few test images
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
# Find the last conv layer in MobileNetV2
last_conv_layer = None
for layer in model.layers[0].layers[::-1]:
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

# Build grad-cam model using the base MobileNetV2 + head
gradcam_base = model.layers[0]

for i in range(3):
    idx = np.random.randint(len(X_test))
    img = X_test[idx]
    img_batch = np.expand_dims(img, axis=0)

    pred_label = np.argmax(model.predict(img_batch, verbose=0))
    true_label = int(y_test[idx])

    heatmap = make_gradcam_heatmap(img_batch, model, last_conv_layer)
    superimposed = display_gradcam(img, heatmap)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Original — True: {LABEL_NAMES[true_label]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(heatmap, cmap="jet")
    axes[i, 1].set_title("Grad-CAM Heatmap")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(superimposed)
    axes[i, 2].set_title(f"Overlay — Pred: {LABEL_NAMES[pred_label]}")
    axes[i, 2].axis("off")

plt.suptitle("Grad-CAM Explainability", fontsize=16)
plt.tight_layout()
plt.show()

---
**Done!** The trained model is saved at `models/cnn_model.h5`. You can now run the Streamlit web app:

```bash
streamlit run app/streamlit_app.py
```